> **Versión corregida (v2.12, ejecución verificada)** — generada a partir de `04_definicion_mdp_limpio.ipynb (Drive/Colab Notebooks)` sin sobrescribirlo.
>
> Cambios aplicados:
> - Celda de inicialización sustituida por el bootstrap portable: las rutas fijas `/content/drive/MyDrive/PIDA-RL-Diabetes` ahora dependen de `PROJECT_ROOT`.
> - Renombrado semántico: el antiguo `PROJECT_ROOT` (copia del repositorio) pasa a `REPO_ROOT`; el antiguo `PERSISTENT_ROOT` (Drive) pasa a `PROJECT_ROOT` (se conserva `PERSISTENT_ROOT` como alias).
> - El clonado usa la rama `feature/entorno-capacidad-v2`; antes clonaba `main`, que no contiene `src/` V2.
> - Se documenta (tabla y especificación JSON) la discrepancia de costos/capacidad entre este entorno histórico y el entorno V2.
>
> Rutas: `PROJECT_ROOT` = datos/modelos/resultados (Drive en Colab, o `PIDA_PROJECT_ROOT`); `REPO_ROOT` = código. Modo prueba: `PIDA_MODO_PRUEBA=1` (cohorte pequeña y pocos episodios).


# PIDA-RL Diabetes - 04. MDP limpio y validado

Este notebook crea una nueva version del entorno bajo el nombre `diabetes_followup_env_clean.py`. No reutiliza ni modifica los modulos anteriores, evitando conflictos con versiones que esperaban `num_comorbilidades`.

> Las transiciones, eventos y recompensas son supuestos transparentes del simulador; no son recomendaciones clinicas.


## 1. Inicializacion y carga de cohorte


In [ ]:
# === Bootstrap portable: PROJECT_ROOT (corrección v2.12) ===
# PROJECT_ROOT = raíz persistente de datos, modelos y resultados.
# REPO_ROOT    = copia local del código (repositorio GitHub).
# Prioridad: variables de entorno PIDA_PROJECT_ROOT / PIDA_REPO_ROOT,
# después Google Colab + Drive y, por último, ejecución local.
from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import importlib
import importlib.util
import subprocess

PROJECT_NAME = 'pida-rl-datos-publicos'

REPO_URL = 'https://github.com/JssSalas/pida-rl-datos-publicos.git'
BRANCH = os.environ.get('PIDA_BRANCH', 'main')
PROJECT_FOLDER_NAME = 'PIDA-RL-Diabetes'
IN_COLAB = 'google.colab' in sys.modules

DRIVE_ROOT = None
if os.environ.get('PIDA_PROJECT_ROOT'):
    PROJECT_ROOT = Path(os.environ['PIDA_PROJECT_ROOT']).expanduser().resolve()
elif IN_COLAB:
    DRIVE_ROOT = next(
        (p for p in [Path('/content/drive'), Path('/content/gdrive')] if (p / 'MyDrive').is_dir()),
        None,
    )
    if DRIVE_ROOT is None:
        from google.colab import drive
        mount_point = Path('/content/drive')
        if mount_point.exists() and any(mount_point.iterdir()):
            mount_point = Path('/content/gdrive')
        drive.mount(str(mount_point), force_remount=False)
        DRIVE_ROOT = mount_point
    PROJECT_ROOT = DRIVE_ROOT / 'MyDrive' / PROJECT_FOLDER_NAME
else:
    PROJECT_ROOT = (Path.cwd() / PROJECT_FOLDER_NAME).resolve()
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)


def _buscar_repo(inicio):
    for candidato in [inicio, *inicio.parents]:
        if (candidato / 'src' / 'environment' / 'diabetes_followup_env.py').is_file():
            return candidato
    return None


if os.environ.get('PIDA_REPO_ROOT'):
    REPO_ROOT = Path(os.environ['PIDA_REPO_ROOT']).expanduser().resolve()
else:
    REPO_ROOT = _buscar_repo(Path.cwd().resolve()) or (
        Path('/content/pida-rl-datos-publicos') if IN_COLAB
        else Path.cwd().resolve() / 'pida-rl-datos-publicos'
    )
if not REPO_ROOT.exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)])
elif not (REPO_ROOT / 'src').is_dir():
    raise RuntimeError(f'REPO_ROOT no contiene la carpeta src: {REPO_ROOT}')
elif IN_COLAB and (REPO_ROOT / '.git').is_dir():
    # En Colab se alinea la copia temporal con la rama de trabajo (sin push ni commit).
    for args in (['fetch', 'origin', BRANCH], ['checkout', BRANCH], ['pull', '--ff-only', 'origin', BRANCH]):
        subprocess.run(['git', '-C', str(REPO_ROOT), *args], check=False)

SRC_ROOT = REPO_ROOT / 'src'
for ruta in (str(REPO_ROOT), str(SRC_ROOT)):
    if ruta not in sys.path:
        sys.path.insert(0, ruta)
os.chdir(REPO_ROOT)

# Estructura persistente común (antes: rutas fijas /content/drive/MyDrive/PIDA-RL-Diabetes).
PERSISTENT_ROOT = PROJECT_ROOT  # alias de compatibilidad con los notebooks originales
MYDRIVE_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_ROOT / 'raw'
INTERIM_DATA_DIR = DATA_ROOT / 'interim'
PROCESSED_DATA_DIR = DATA_ROOT / 'processed'
METADATA_DIR = DATA_ROOT / 'metadata'
SPECIFICATIONS_DIR = DATA_ROOT / 'specifications'
MODELS_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results'
COHORT_PATH = PROCESSED_DATA_DIR / 'cohorte_diabetes.csv'

# Modo prueba: cohorte pequeña y pocos episodios (PIDA_MODO_PRUEBA=1).
MODO_PRUEBA = os.environ.get('PIDA_MODO_PRUEBA', '0') == '1'
N_COHORTE_PRUEBA = int(os.environ.get('PIDA_N_COHORTE', '120'))
SEMILLA_MUESTRA_PRUEBA = 2026
RESULTS_V2_DIR = RESULTS_DIR / ('capacidad_v2_prueba' if MODO_PRUEBA else 'capacidad_v2')


def muestra_cohorte_prueba(cohorte, n=None, semilla=SEMILLA_MUESTRA_PRUEBA):
    """Devuelve (cohorte, factor). En modo prueba toma una submuestra
    estratificada por categoria_riesgo (bajo/medio/alto); no crea columnas."""
    n = N_COHORTE_PRUEBA if n is None else int(n)
    if not MODO_PRUEBA or len(cohorte) <= n:
        return cohorte, 1.0
    fraccion = n / len(cohorte)
    muestra = (
        cohorte.groupby('categoria_riesgo', group_keys=False)
        .sample(frac=fraccion, random_state=semilla)
        .sort_values('FOLIO_INT')
        .reset_index(drop=True)
    )
    return muestra, len(muestra) / len(cohorte)


def escalar_capacidad(capacidad, factor):
    """Mantiene la presión de capacidad (recursos/persona) al reducir la cohorte."""
    return capacidad if factor >= 1.0 else max(1, int(round(capacidad * factor)))

print('Entorno:', 'Google Colab' if IN_COLAB else 'local')
print('PROJECT_ROOT (datos/resultados):', PROJECT_ROOT)
print('REPO_ROOT (código):', REPO_ROOT)
print('Modo prueba:', MODO_PRUEBA)

if not COHORT_PATH.is_file():
    raise FileNotFoundError(f'No existe la cohorte: {COHORT_PATH}. Ejecuta primero el notebook 03.')
METADATA_DIR.mkdir(parents=True, exist_ok=True)
print('Cohorte:', COHORT_PATH)


In [ ]:
import numpy as np
import pandas as pd

df_cohorte_diabetes = pd.read_csv(COHORT_PATH, dtype={'FOLIO_INT': 'string'})

required_columns = [
    'FOLIO_INT', 'edad', 'sexo', 'anios_con_diabetes',
    'insulina_diaria', 'depresion', 'consultas_control_12m',
    'num_hospitalizaciones', 'num_complicaciones',
    'score_riesgo', 'categoria_riesgo'
]

missing = [column for column in required_columns if column not in df_cohorte_diabetes.columns]
if missing:
    raise KeyError('Faltan columnas en cohorte: ' + ', '.join(missing))

numeric_columns = [
    'edad', 'sexo', 'anios_con_diabetes', 'insulina_diaria',
    'depresion', 'consultas_control_12m', 'num_hospitalizaciones',
    'num_complicaciones', 'score_riesgo'
]

assert np.isfinite(df_cohorte_diabetes[numeric_columns].to_numpy(dtype=float)).all()
assert df_cohorte_diabetes['categoria_riesgo'].isin(['bajo', 'medio', 'alto']).all()
assert not df_cohorte_diabetes['FOLIO_INT'].duplicated().any()

print('Cohorte valida:', df_cohorte_diabetes.shape)
display(df_cohorte_diabetes.head())


## 2. Crear un entorno con nombre nuevo

Este archivo usa solo variables disponibles en la cohorte actual. No contiene `num_comorbilidades`.


In [ ]:
ENV_DIR = SRC_ROOT / 'environment'
ENV_DIR.mkdir(parents=True, exist_ok=True)
(ENV_DIR / '__init__.py').touch(exist_ok=True)
CLEAN_ENV_FILE = ENV_DIR / 'diabetes_followup_env_clean.py'

clean_env_code = '''import numpy as np
import gymnasium as gym
from gymnasium import spaces


class DiabetesFollowUpEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, cohorte_df, capacidad_mensual=0.20, horizonte=12, seed=None):
        super().__init__()
        required = [
            "edad", "sexo", "anios_con_diabetes",
            "insulina_diaria", "depresion", "consultas_control_12m",
            "num_hospitalizaciones", "num_complicaciones",
            "score_riesgo", "categoria_riesgo"
        ]
        missing = [name for name in required if name not in cohorte_df.columns]
        if missing:
            raise KeyError("Faltan columnas: " + ", ".join(missing))
        if not 0 < float(capacidad_mensual) <= 1:
            raise ValueError("capacidad_mensual debe estar entre 0 y 1.")
        if int(horizonte) < 1:
            raise ValueError("horizonte debe ser al menos 1.")
        self.cohorte_base = cohorte_df.reset_index(drop=True).copy()
        self.n_perfiles = len(self.cohorte_base)
        if self.n_perfiles == 0:
            raise ValueError("La cohorte no puede estar vacia.")
        numeric = [
            "edad", "sexo", "anios_con_diabetes",
            "insulina_diaria", "depresion", "consultas_control_12m",
            "num_hospitalizaciones", "num_complicaciones", "score_riesgo"
        ]
        if not np.isfinite(self.cohorte_base[numeric].to_numpy(dtype=float)).all():
            raise ValueError("La cohorte contiene NaN o infinitos.")
        if not self.cohorte_base["categoria_riesgo"].isin({"bajo", "medio", "alto"}).all():
            raise ValueError("categoria_riesgo invalida.")
        self.capacidad_mensual = float(capacidad_mensual)
        self.horizonte = int(horizonte)
        self.action_space = spaces.Discrete(4)
        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(13,), dtype=np.float32)
        self.costo_accion = {0: 0, 1: 1, 2: 2, 3: 3}
        self.reduccion_evento = {0: 0.00, 1: 0.20, 2: 0.45, 3: 0.65}
        self.riesgo_ordinal = {"bajo": 0, "medio": 1, "alto": 2}
        self.reset(seed=seed)

    @staticmethod
    def _norm(value, denominator):
        return float(np.clip(float(value) / max(float(denominator), 1.0), 0.0, 1.0))

    def _risk_level(self, category):
        return self.riesgo_ordinal[str(category)]

    def _get_obs(self, idx):
        row = self.cohorte.iloc[idx]
        remaining = self.presupuesto_mensual - self.recursos_usados_mes
        sex_norm = 0.0 if float(row["sexo"]) == 1.0 else 1.0
        risk_norm = self._risk_level(row["categoria_riesgo"]) / 2.0
        return np.array([
            self._norm(row["edad"], 100),
            sex_norm,
            self._norm(row["anios_con_diabetes"], 80),
            float(np.clip(row["insulina_diaria"], 0, 1)),
            float(np.clip(row["depresion"], 0, 1)),
            self._norm(row["consultas_control_12m"], 24),
            self._norm(row["num_hospitalizaciones"], 12),
            self._norm(row["num_complicaciones"], 5),
            self._norm(row["score_riesgo"], 16),
            risk_norm,
            self._norm(row["meses_sin_contacto"], self.horizonte),
            self._norm(self.mes_actual, self.horizonte),
            self._norm(remaining, self.presupuesto_mensual)
        ], dtype=np.float32)

    def _event_probability(self, row, risk_level):
        value = (
            0.01 + 0.025 * risk_level
            + 0.010 * float(row["num_complicaciones"])
            + 0.020 * float(row["num_hospitalizaciones"] > 0)
            + 0.005 * min(float(row["meses_sin_contacto"]), 6)
        )
        return float(np.clip(value, 0.0, 0.80))

    def _reward(self, action, risk_level, action_accepted, adverse_event, over_capacity):
        reward = 0.0
        if action_accepted and action > 0:
            reward += {1: 0.5, 2: 1.0, 3: 1.5}[action] * (risk_level + 1)
        if action == 0 or not action_accepted:
            reward -= {0: 0.0, 1: 0.5, 2: 1.5}[risk_level]
        if adverse_event:
            reward -= 5.0
        if over_capacity:
            reward -= 3.0
        return float(reward)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.cohorte = self.cohorte_base.copy()
        self.cohorte["meses_sin_contacto"] = 0
        self.mes_actual = 1
        self.idx_actual = 0
        self.presupuesto_mensual = max(1, int(round(self.n_perfiles * self.capacidad_mensual)))
        self.recursos_usados_mes = 0
        self.contactos_aceptados_mes = 0
        self.intentos_exceso_mes = 0
        obs = self._get_obs(self.idx_actual)
        info = {
            "mes": self.mes_actual,
            "indice_perfil": self.idx_actual,
            "n_perfiles": self.n_perfiles,
            "presupuesto_mensual": self.presupuesto_mensual,
            "recursos_usados_mes": 0,
            "recursos_restantes_mes": self.presupuesto_mensual
        }
        return obs, info

    def step(self, action):
        action = int(action)
        if not self.action_space.contains(action):
            raise ValueError(f"Accion invalida: {action}")
        idx = self.idx_actual
        month = self.mes_actual
        row = self.cohorte.iloc[idx]
        risk_level = self._risk_level(row["categoria_riesgo"])
        requested_cost = self.costo_accion[action]
        over_capacity = self.recursos_usados_mes + requested_cost > self.presupuesto_mensual
        action_accepted = not over_capacity
        actual_cost = requested_cost if action_accepted else 0
        if over_capacity:
            self.intentos_exceso_mes += 1
        else:
            self.recursos_usados_mes += actual_cost
            if action > 0:
                self.contactos_aceptados_mes += 1
        base_probability = self._event_probability(row, risk_level)
        reduction = self.reduccion_evento[action] if action_accepted else 0.0
        probability = float(np.clip(base_probability * (1.0 - reduction), 0.0, 1.0))
        adverse_event = bool(self.np_random.random() < probability)
        reward = self._reward(action, risk_level, action_accepted, adverse_event, over_capacity)
        if action > 0 and action_accepted:
            self.cohorte.loc[idx, "meses_sin_contacto"] = 0
        else:
            self.cohorte.loc[idx, "meses_sin_contacto"] += 1
        used = int(self.recursos_usados_mes)
        remaining = int(self.presupuesto_mensual - self.recursos_usados_mes)
        contacts = int(self.contactos_aceptados_mes)
        excess_attempts = int(self.intentos_exceso_mes)
        self.idx_actual += 1
        terminated = False
        truncated = False
        month_changed = False
        if self.idx_actual >= self.n_perfiles:
            self.idx_actual = 0
            self.mes_actual += 1
            month_changed = True
            self.recursos_usados_mes = 0
            self.contactos_aceptados_mes = 0
            self.intentos_exceso_mes = 0
            if self.mes_actual > self.horizonte:
                terminated = True
        obs = np.zeros(self.observation_space.shape, dtype=np.float32) if terminated else self._get_obs(self.idx_actual)
        info = {
            "mes": month,
            "mes_siguiente": self.mes_actual,
            "cambio_mes": month_changed,
            "indice_perfil": idx,
            "nivel_riesgo": risk_level,
            "categoria_riesgo": str(row["categoria_riesgo"]),
            "evento_adverso": adverse_event,
            "prob_evento": probability,
            "excede_capacidad": over_capacity,
            "accion_aceptada": action_accepted,
            "costo_solicitado": requested_cost,
            "costo_real": actual_cost,
            "presupuesto_mensual": self.presupuesto_mensual,
            "recursos_usados_mes": used,
            "recursos_restantes_mes": remaining,
            "contactos_aceptados_mes": contacts,
            "intentos_exceso_mes": excess_attempts
        }
        return obs, reward, terminated, truncated, info
'''

CLEAN_ENV_FILE.write_text(clean_env_code, encoding='utf-8')

content = CLEAN_ENV_FILE.read_text(encoding='utf-8')
assert 'num_comorbilidades' not in content
assert 'depresion' in content
assert 'shape=(13,)' in content

print('Entorno limpio creado:', CLEAN_ENV_FILE)
print('Tamano:', CLEAN_ENV_FILE.stat().st_size, 'bytes')


## 3. Importacion sin cache y validacion


In [ ]:
for module_name in list(sys.modules):
    if module_name.startswith('environment.diabetes_followup_env_clean'):
        del sys.modules[module_name]

importlib.invalidate_caches()

import environment.diabetes_followup_env_clean as clean_module
DiabetesFollowUpEnv = clean_module.DiabetesFollowUpEnv

print('Modulo importado desde:', clean_module.__file__)
assert clean_module.__file__.endswith('diabetes_followup_env_clean.py')

try:
    import gymnasium
    import stable_baselines3
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'gymnasium', 'stable-baselines3'])

from stable_baselines3.common.env_checker import check_env

env_check = DiabetesFollowUpEnv(df_cohorte_diabetes, capacidad_mensual=0.20, horizonte=12, seed=42)
check_env(env_check, warn=True)

obs, info = env_check.reset(seed=42)
assert obs.shape == (13,)
assert env_check.observation_space.contains(obs)
assert np.isfinite(obs).all()

print('Entorno limpio validado.')
print('Observacion:', env_check.observation_space)
print('Acciones:', env_check.action_space)
print('Presupuesto mensual:', env_check.presupuesto_mensual)


## 4. Pruebas funcionales


In [ ]:
env_test = DiabetesFollowUpEnv(df_cohorte_diabetes, capacidad_mensual=0.20, horizonte=2, seed=2026)
obs, info = env_test.reset(seed=2026)

for step in range(250):
    action = env_test.action_space.sample()
    obs, reward, terminated, truncated, info = env_test.step(action)
    assert np.isfinite(reward), f'Recompensa no finita en paso {step}'
    assert np.isfinite(obs).all(), f'Observacion no finita en paso {step}'
    assert env_test.observation_space.contains(obs), f'Observacion fuera de espacio en paso {step}'
    assert info['recursos_usados_mes'] <= info['presupuesto_mensual']
    assert info['recursos_restantes_mes'] >= 0
    if terminated or truncated:
        break

env_capacity = DiabetesFollowUpEnv(df_cohorte_diabetes, capacidad_mensual=0.01, horizonte=1, seed=42)
obs, info = env_capacity.reset(seed=42)
rejections = 0

while True:
    obs, reward, terminated, truncated, info = env_capacity.step(3)
    rejections += int(info['excede_capacidad'])
    assert info['recursos_usados_mes'] <= info['presupuesto_mensual']
    if terminated or truncated:
        break

assert rejections > 0
print('Pruebas funcionales correctas.')
print('Solicitudes rechazadas por capacidad:', rejections)


## 5. Guardar especificacion tecnica


In [ ]:
MDP_SPEC_PATH = METADATA_DIR / 'especificacion_mdp_clean_v1.json'

spec = {
    'version': 'clean_v1',
    'environment_module': 'environment.diabetes_followup_env_clean',
    'environment_file': str(CLEAN_ENV_FILE),
    'cohort_path': str(COHORT_PATH),
    'n_perfiles': int(len(df_cohorte_diabetes)),
    'state_dimension': 13,
    'actions': {
        '0': {'nombre': 'sin_contacto', 'costo': 0},
        '1': {'nombre': 'mensaje', 'costo': 1},
        '2': {'nombre': 'llamada', 'costo': 2},
        '3': {'nombre': 'teleorientacion', 'costo': 3}
    },
    'scope_note': 'Las transiciones y recompensas son supuestos del simulador.',
    'nota_v2': ('Entorno histórico de referencia (costos 0/1/2/3 y capacidad como fracción 0.20). '
                'Los notebooks 05-09 usan src/environment/diabetes_followup_env.py: costos 0/1/3/5, '
                'capacidad absoluta y degradación de la acción en lugar de rechazo.')
}

with MDP_SPEC_PATH.open('w', encoding='utf-8') as file:
    json.dump(spec, file, ensure_ascii=False, indent=2)

print('Especificacion guardada:', MDP_SPEC_PATH)
print('Existe:', MDP_SPEC_PATH.is_file())


## Discrepancia documentada con el entorno V2

| Aspecto | Entorno 04 (`diabetes_followup_env_clean`) | Entorno V2 (`src/environment/diabetes_followup_env.py`, notebooks 05–09) |
|---|---|---|
| Costos por acción | 0 / 1 / 2 / 3 | 0 / 1 / 3 / 5 |
| Capacidad | Fracción de la cohorte (0.20) | Unidades absolutas por mes (100) |
| Exceso de capacidad | Rechaza la acción y penaliza −3 | Ejecuta la acción viable más intensa (degradación) |
| Estado | 13 variables normalizadas | 13 variables normalizadas (mismas columnas de cohorte) |
| Categorías de riesgo | bajo / medio / alto | bajo / medio / alto |

Ambos usan exactamente las columnas de `cohorte_diabetes.csv`. El 04 se conserva como validación histórica del MDP; los resultados comparativos oficiales provienen del entorno V2. La acción 3 es teleorientación, no una categoría de riesgo.


## Resultado esperado

Debe mostrarse `Entorno limpio validado.` y `Pruebas funcionales correctas.`. A continuacion utiliza solo el notebook 05 limpio correspondiente.
